# CS F425 Deep Learning Project — Phase 2
## ToolAlpaca SFT and ReAct Execution Loop

Extracted from `CS_F425_DL_Project.ipynb`.

This notebook keeps the original Phase 2 section and includes the shared setup cells it depends on: imports, Drive mounting, data loading, action parsing, answer computation, and evaluation helpers.


## Cell 1 — Install Packages

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

DEPS_SENTINEL = Path("/tmp/.csf425_phi2_deps_ready")
PACKAGES = [
    "transformers>=4.41.0",
    "peft>=0.10.0",
    "trl>=0.9.0",
    "accelerate>=0.29.3",
    "datasets>=2.19.0",
    "bitsandbytes>=0.44.0",
    "einops",
]

if not DEPS_SENTINEL.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *PACKAGES])
    DEPS_SENTINEL.write_text("ok")
    print("Dependencies installed or upgraded. Restarting the Colab runtime once...")
    os.kill(os.getpid(), 9)

print("Dependencies already prepared for this runtime.")


In [ ]:
# Cell 1 restarts the Colab runtime once after upgrading packages.
# After the reconnect, continue from the next cell.


## Cell 2 — Imports & GPU Check

In [ ]:
import json
import random
import re

import pandas as pd
import torch
from datasets import Dataset
from packaging import version
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

try:
    import trl
except Exception as exc:
    raise RuntimeError(
        "Failed to import trl. Re-run Cell 1 and let Colab restart once."
    ) from exc

if not hasattr(trl, "SFTTrainer"):
    raise RuntimeError(
        "trl.SFTTrainer is unavailable. Re-run Cell 1 and let Colab restart once."
    )

SFTTrainer = trl.SFTTrainer
SFTConfig = getattr(trl, "SFTConfig", TrainingArguments)
_USE_SFTCONFIG = hasattr(trl, "SFTConfig")

import transformers as _tf

if version.parse(_tf.__version__) < version.parse("4.41.0"):
    raise RuntimeError(
        f"transformers=={_tf.__version__} is too old for this notebook. "
        "Re-run Cell 1 and let Colab restart once."
    )

print(f"trl         : {trl.__version__}  (SFTConfig available: {_USE_SFTCONFIG})")
print(f"transformers: {_tf.__version__}")

assert torch.cuda.is_available(), "No GPU. In Colab, switch to a T4 GPU runtime."
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Cell 3 — Mount Google Drive

Saves adapter weights and checkpoints to Drive so they persist across Colab sessions.

**One-time setup**: Upload these files to `MyDrive/CS_F425_Project/`:
- `sales_data.csv`
- `agent_trajectories_2k.json`
- `tool_executor.py`
- `run_pipeline.py`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, glob as _glob
DRIVE_DIR = "/content/drive/MyDrive/CS_F425_Project"
ADAPTER_DIR = f"{DRIVE_DIR}/phi2-agent-adapter"
CKPT_DIR = f"{DRIVE_DIR}/phi2-agent-qlora"

os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# Make tool_executor.py importable
sys.path.insert(0, DRIVE_DIR)


def detect_training_state():
    """Detect what checkpoints are available and decide the execution path.

    Returns (state, best_path) where state is one of:
      TRAINING_COMPLETE  - final adapter saved (skip training, go to inference)
      EPOCH_CHECKPOINT   - HF Trainer checkpoint exists (resume training)
      TIMED_CHECKPOINT   - timed adapter-only snapshot (skip training, inference-capable)
      FRESH              - nothing found (train from scratch)

    Checkpoints are sorted by their numeric suffix (highest number wins),
    so deleting earlier checkpoints to save space will not cause issues.
    """
    # 1. Final adapter saved by Cell 13?
    if os.path.isfile(f"{ADAPTER_DIR}/adapter_config.json"):
        return "TRAINING_COMPLETE", ADAPTER_DIR

    # 2. Trainer epoch checkpoints (contain full optimizer state)?
    epoch_ckpts = _glob.glob(f"{CKPT_DIR}/checkpoint-*")
    if epoch_ckpts:
        epoch_ckpts.sort(key=lambda p: int(p.rsplit("-", 1)[-1]))
        return "EPOCH_CHECKPOINT", epoch_ckpts[-1]

    # 3. Timed adapter-only snapshots?
    timed_ckpts = _glob.glob(f"{ADAPTER_DIR}/timed_ckpt_step_*")
    if timed_ckpts:
        timed_ckpts.sort(key=lambda p: int(p.rsplit("_", 1)[-1]))
        return "TIMED_CHECKPOINT", timed_ckpts[-1]

    return "FRESH", None


TRAINING_STATE, BEST_CKPT_PATH = detect_training_state()
SKIP_TRAINING = TRAINING_STATE in ("TRAINING_COMPLETE", "TIMED_CHECKPOINT")

print(f"Drive dir       : {DRIVE_DIR}")
print(f"Contents        : {os.listdir(DRIVE_DIR)}")
print(f"Training state  : {TRAINING_STATE}")
print(f"Best checkpoint : {BEST_CKPT_PATH}")
print(f"Skip training   : {SKIP_TRAINING}")

## Cell 4 — Load Data

In [ ]:
df = pd.read_csv(f"{DRIVE_DIR}/sales_data.csv")
print(f"sales_data shape: {df.shape}")
print(df.dtypes)
print(df.head(3))

with open(f"{DRIVE_DIR}/agent_trajectories_2k.json") as f:
    trajectories = json.load(f)
print(f"\nTrajectories loaded: {len(trajectories)}")
print("Sample entry:", json.dumps(trajectories[0], indent=2))

## Cell 5 — Pre-compute Answers

The training data has `query` + `actions` but **no answers**.  
We execute each action sequence against `sales_data.csv` using `ToolExecutor` to obtain the gold answer.

We also extend `parse_agent_action` to handle `aggregate_mean`, `aggregate_count`, and string filter values that the original `run_pipeline.py` misses.

In [ ]:
import os
from numbers import Integral, Real

from tool_executor import ToolExecutor


def _ensure_phase2_source_data():
    """Load sales_data.csv and agent_trajectories_2k.json if they are missing from memory."""
    global df, trajectories

    sales_path = None
    traj_path = None
    search_roots = []
    if "DRIVE_DIR" in globals():
        search_roots.append(DRIVE_DIR)
    search_roots.append(os.getcwd())

    for root in search_roots:
        candidate_sales = os.path.join(root, "sales_data.csv")
        candidate_traj = os.path.join(root, "agent_trajectories_2k.json")
        if sales_path is None and os.path.isfile(candidate_sales):
            sales_path = candidate_sales
        if traj_path is None and os.path.isfile(candidate_traj):
            traj_path = candidate_traj

    if ("df" not in globals() or df is None) and sales_path is not None:
        df = pd.read_csv(sales_path)
        print(f"Loaded sales_data.csv from: {sales_path}")

    if ("trajectories" not in globals() or trajectories is None) and traj_path is not None:
        with open(traj_path, "r", encoding="utf-8") as f:
            trajectories = json.load(f)
        print(f"Loaded agent_trajectories_2k.json from: {traj_path}")

    if "df" not in globals() or df is None:
        raise RuntimeError("sales_data.csv is not loaded. Run the data-load cell or place the file in DRIVE_DIR/current working directory.")
    if "trajectories" not in globals() or trajectories is None:
        raise RuntimeError("agent_trajectories_2k.json is not loaded. Run the data-load cell or place the file in DRIVE_DIR/current working directory.")

    return df, trajectories


def _parse_numeric(val_str):
    """Convert a string to int or float as appropriate."""
    return float(val_str) if '.' in val_str else int(val_str)


def parse_agent_action(action_str):
    """Parse one string-form action into ToolExecutor format."""
    if action_str.startswith("filter_data"):
        # Try numeric value first (int or float, possibly negative)
        m = re.search(r"column='([^']+)',\s*value=(-?[\d]+(?:\.[\d]+)?)", action_str)
        if m:
            return {
                "tool": "filter",
                "args": {"column": m.group(1), "op": "==", "value": _parse_numeric(m.group(2))},
            }

        # Try string value
        m = re.search(r"column='([^']+)',\s*value='([^']+)'", action_str)
        if m:
            return {
                "tool": "filter",
                "args": {"column": m.group(1), "op": "==", "value": m.group(2)},
            }

    elif action_str.startswith("group_by"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "groupby", "args": {"column": m.group(1)}}

    elif action_str.startswith("aggregate_sum"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "sum"}}

    elif action_str.startswith("aggregate_mean"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "mean"}}

    elif action_str.startswith("aggregate_count"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "count"}}

    elif action_str.startswith("sort_by"):
        m = re.search(r"column='([^']+)',\s*order='([^']+)'", action_str)
        if m:
            return {
                "tool": "sort",
                "args": {"column": m.group(1), "ascending": m.group(2) != "desc"},
            }

    elif action_str.startswith("top_k"):
        m = re.search(r"k=(\d+)", action_str)
        if m:
            return {"tool": "topk", "args": {"k": int(m.group(1))}}

    return None


def clean_scalar(value):
    if hasattr(value, "item"):
        try:
            value = value.item()
        except Exception:
            pass

    if isinstance(value, bool):
        return bool(value)
    if isinstance(value, Integral):
        return int(value)
    if isinstance(value, Real):
        return round(float(value), 4)
    return value


def result_to_python(actions, result_df):
    """Convert ToolExecutor output into the JSON shape used for supervision."""
    if result_df is None or (hasattr(result_df, "empty") and result_df.empty):
        return None

    action_names = [action.split("(")[0] for action in actions]

    # Single scalar result
    if result_df.shape == (1, 1):
        return clean_scalar(result_df.iloc[0, 0])

    # group_by + aggregate (without sort/topk) → key-value dict
    has_groupby = any(a.startswith("group_by") for a in action_names)
    has_sort_or_topk = any(
        a.startswith("sort_by") or a.startswith("top_k") for a in action_names
    )
    if result_df.shape[1] == 2 and has_groupby and not has_sort_or_topk:
        key_col, value_col = result_df.columns
        return {
            str(row[key_col]): clean_scalar(row[value_col])
            for _, row in result_df.iterrows()
        }

    # Default: list of dicts (for sorted/topk results or multi-column outputs)
    return [
        {str(key): clean_scalar(value) for key, value in row.items()}
        for row in result_df.to_dict(orient="records")
    ]


def compute_answer(actions, df):
    """Parse and execute a list of action strings."""
    parsed = []
    for action in actions:
        try:
            parsed_action = parse_agent_action(action)
        except Exception:
            parsed_action = None

        if parsed_action is not None:
            parsed.append(parsed_action)

    if not parsed:
        return None

    try:
        result = ToolExecutor(df.copy()).execute(parsed)
        return result_to_python(actions, result)
    except Exception:
        return None


df, trajectories = _ensure_phase2_source_data()
training_data = []
skipped = 0

for entry in trajectories:
    answer = compute_answer(entry["actions"], df)
    if answer is None:
        skipped += 1
        continue

    output_json = json.dumps(
        {"actions": entry["actions"], "answer": answer},
        ensure_ascii=False,
    )
    training_data.append({"query": entry["query"], "output_json": output_json})

print(f"Valid examples : {len(training_data)} / {len(trajectories)}")
print(f"Skipped        : {skipped}")
print("\nSample:")
print(json.dumps(json.loads(training_data[0]["output_json"]), indent=2, ensure_ascii=False))

## Shared Evaluation Helper


In [ ]:
def normalize_answer(value):
    if isinstance(value, float):
        return round(value, 4)
    if isinstance(value, list):
        return [normalize_answer(item) for item in value]
    if isinstance(value, dict):
        return {key: normalize_answer(val) for key, val in value.items()}
    return value


## Cell 17 — Phase 2 Config & Paths

In [ ]:
# -- Phase 2 configuration --
MODEL_NAME_P2 = "mistralai/Mistral-7B-v0.1"  # Keep a 7B base model to match the project spec.
# Use fresh output folders so the notebook does not silently reuse older slow checkpoints.
ADAPTER_DIR_P2 = f"{DRIVE_DIR}/mistral-react-adapter-fast-nogc"
CKPT_DIR_P2 = f"{DRIVE_DIR}/mistral-react-qlora-fast-nogc"
P2_WARMUP_ADAPTER_DIR = f"{DRIVE_DIR}/mistral-react-toolalpaca-warmup-fast-nogc"
P2_WARMUP_CKPT_DIR = f"{DRIVE_DIR}/mistral-react-toolalpaca-warmup-ckpt-fast-nogc"
P2_TOOLALPACA_CACHE_PATH = f"{DRIVE_DIR}/toolalpaca_train_data.json"
P2_TOOLALPACA_URL = "https://raw.githubusercontent.com/tangqiaoyu/ToolAlpaca/main/data/train_data.json"

# Fast-mode defaults tuned for free-tier T4 runs.
P2_FAST_MODE = True
MAX_SEQ_LENGTH_P2 = 512 if P2_FAST_MODE else 768
P2_OBSERVATION_CHAR_LIMIT = 320 if P2_FAST_MODE else 500
P2_MAX_TRAIN_EXAMPLES = 900 if P2_FAST_MODE else 1800
P2_MAX_VALID_EXAMPLES = 100 if P2_FAST_MODE else 200
P2_NUM_EPOCHS = 1 if P2_FAST_MODE else 2
P2_BATCH_SIZE = 1 if P2_FAST_MODE else 2
P2_GRAD_ACCUM = 16 if P2_FAST_MODE else 8
P2_USE_GRADIENT_CHECKPOINTING = False if P2_FAST_MODE else True
P2_LEARNING_RATE = 2e-4
P2_LOGGING_STEPS = 10 if P2_FAST_MODE else 25
P2_ENABLE_TRAIN_EVAL = False if P2_FAST_MODE else True
P2_LORA_R = 8 if P2_FAST_MODE else 16
P2_LORA_ALPHA = 16 if P2_FAST_MODE else 32
P2_LORA_DROPOUT = 0.05
P2_LORA_TARGET_MODULES = ["q_proj", "v_proj"] if P2_FAST_MODE else ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Mandatory ToolAlpaca warm-up, but capped for speed.
P2_WARMUP_REQUIRED = True
P2_WARMUP_MAX_EXAMPLES = 192 if P2_FAST_MODE else 512
P2_WARMUP_NUM_EPOCHS = 1
P2_WARMUP_MAX_STEPS_PER_INSTANCE = 3 if P2_FAST_MODE else 5
P2_WARMUP_DOC_CHAR_LIMIT = 1200 if P2_FAST_MODE else 2000

os.makedirs(ADAPTER_DIR_P2, exist_ok=True)
os.makedirs(CKPT_DIR_P2, exist_ok=True)
os.makedirs(P2_WARMUP_ADAPTER_DIR, exist_ok=True)
os.makedirs(P2_WARMUP_CKPT_DIR, exist_ok=True)

# Detect Phase 2 training state. Only the finalized adapter means training is done;
# timed adapter snapshots are training resume points, not skip signals.
def detect_p2_state():
    if os.path.isfile(f"{ADAPTER_DIR_P2}/adapter_config.json"):
        return "TRAINING_COMPLETE", ADAPTER_DIR_P2
    epoch_ckpts = _glob.glob(f"{CKPT_DIR_P2}/checkpoint-*")
    if epoch_ckpts:
        epoch_ckpts.sort(key=lambda p: int(p.rsplit("-", 1)[-1]))
        return "EPOCH_CHECKPOINT", epoch_ckpts[-1]
    timed_ckpts = _glob.glob(f"{ADAPTER_DIR_P2}/timed_ckpt_step_*")
    if timed_ckpts:
        timed_ckpts.sort(key=lambda p: int(p.rsplit("_", 1)[-1]))
        return "TIMED_CHECKPOINT", timed_ckpts[-1]
    return "FRESH", None

P2_WARMUP_READY = os.path.isfile(f"{P2_WARMUP_ADAPTER_DIR}/adapter_config.json")
P2_STATE, P2_CKPT_PATH = detect_p2_state()
P2_SKIP_TRAINING = P2_STATE == "TRAINING_COMPLETE"

print(f"Phase 2 model          : {MODEL_NAME_P2}")
print(f"Fast mode              : {P2_FAST_MODE}")
print(f"Phase 2 seq length     : {MAX_SEQ_LENGTH_P2}")
print(f"Phase 2 max train ex   : {P2_MAX_TRAIN_EXAMPLES}")
print(f"Phase 2 epochs         : {P2_NUM_EPOCHS}")
print(f"Phase 2 batch / accum  : {P2_BATCH_SIZE} / {P2_GRAD_ACCUM}")
print(f"Grad checkpointing     : {P2_USE_GRADIENT_CHECKPOINTING}")
print(f"Phase 2 LoRA targets   : {P2_LORA_TARGET_MODULES}")
print(f"ToolAlpaca warm-up     : required={P2_WARMUP_REQUIRED}, ready={P2_WARMUP_READY}")
print(f"ToolAlpaca warm-up ex  : {P2_WARMUP_MAX_EXAMPLES}")
print(f"Phase 2 state          : {P2_STATE}")
print(f"Phase 2 ckpt           : {P2_CKPT_PATH}")
print(f"Skip P2 training       : {P2_SKIP_TRAINING}")


## Cell 18 — Build ReAct Training Data

Converts each trajectory into a multi-turn Thought → Action → Action Input → Observation trace by executing actions step-by-step against `ToolExecutor`.

In [ ]:
REACT_SYSTEM_PROMPT = """You are a data analysis agent. You have access to the following tools to analyze a sales dataset:

Function Descriptions:
[
  {"name": "filter_data", "description": "Filter rows where column equals value", "parameters": {"column": "str", "value": "str or int"}},
  {"name": "group_by", "description": "Group the data by a column", "parameters": {"column": "str"}},
  {"name": "aggregate_sum", "description": "Sum a numeric column", "parameters": {"column": "str"}},
  {"name": "aggregate_mean", "description": "Average a numeric column", "parameters": {"column": "str"}},
  {"name": "aggregate_count", "description": "Count rows for a column", "parameters": {"column": "str"}},
  {"name": "sort_by", "description": "Sort by a column", "parameters": {"column": "str", "order": "asc or desc"}},
  {"name": "top_k", "description": "Select top k rows", "parameters": {"k": "int"}}
]

Schema: date (date), year (int), month (int), city (str), region (str), product (str), category (str), revenue (float), units_sold (int), cost (float), profit (float)

Use the following exact format:
Thought: <reasoning>
Action: <tool_name>
Action Input: {"arg": value}

Rules:
- Action Input must be valid compact JSON with double-quoted keys.
- Do not invent Observation text; wait for the real Observation.
- Use only the listed tools.
- End with Final Answer: once you have enough information."""

REACT_PROMPT_TEMPLATE = """### System
{system}

### User Query
{question}

### Agent Scratchpad
{scratchpad}"""


# Thought generation templates.
_THOUGHT_TEMPLATES = {
    "filter_data": "I need to filter the data by {args} to narrow down the dataset.",
    "group_by": "I should group the data by {args} to organize it.",
    "aggregate_sum": "I need to compute the sum of {args}.",
    "aggregate_mean": "I need to compute the average of {args}.",
    "aggregate_count": "I need to count the entries for {args}.",
    "sort_by": "I should sort the results by {args}.",
    "top_k": "I need to select the top {args} entries.",
}


def _extract_args_text(action_str):
    """Extract human-readable args from an action string."""
    m = re.search(r"\((.+)\)", action_str)
    return m.group(1) if m else action_str


def _action_payload_from_action_string(action_str):
    """Convert legacy action syntax into the JSON payload used by Phase 2."""
    parsed = parse_agent_action(action_str)
    if parsed is None:
        return None

    action_name = action_str.split("(", 1)[0]
    args = parsed["args"]

    if action_name == "filter_data":
        return {"column": args["column"], "value": args["value"]}
    if action_name in {"group_by", "aggregate_sum", "aggregate_mean", "aggregate_count"}:
        return {"column": args["column"]}
    if action_name == "sort_by":
        return {
            "column": args["column"],
            "order": "asc" if args["ascending"] else "desc",
        }
    if action_name == "top_k":
        return {"k": int(args["k"])}
    return None


def _normalize_action_payload(action_name, payload):
    """Validate and normalize a parsed Action Input payload."""
    if not isinstance(payload, dict):
        raise TypeError(f"Action Input for {action_name} must be a JSON object.")

    if action_name == "filter_data":
        if "column" not in payload or "value" not in payload:
            raise ValueError("filter_data requires 'column' and 'value'.")
        return {"column": str(payload["column"]), "value": payload["value"]}

    if action_name in {"group_by", "aggregate_sum", "aggregate_mean", "aggregate_count"}:
        if "column" not in payload:
            raise ValueError(f"{action_name} requires 'column'.")
        return {"column": str(payload["column"])}

    if action_name == "sort_by":
        if "column" not in payload or "order" not in payload:
            raise ValueError("sort_by requires 'column' and 'order'.")
        order = str(payload["order"]).lower()
        if order not in {"asc", "desc"}:
            raise ValueError("sort_by order must be 'asc' or 'desc'.")
        return {"column": str(payload["column"]), "order": order}

    if action_name == "top_k":
        if "k" not in payload:
            raise ValueError("top_k requires 'k'.")
        return {"k": int(payload["k"])}

    raise ValueError(f"Unsupported action: {action_name}")


def _action_string_from_payload(action_name, payload):
    """Convert a normalized JSON payload back into the canonical action string."""
    payload = _normalize_action_payload(action_name, payload)

    if action_name == "filter_data":
        value = payload["value"]
        value_repr = repr(value) if isinstance(value, str) else value
        return f"filter_data(column='{payload['column']}', value={value_repr})"

    if action_name == "group_by":
        return f"group_by(column='{payload['column']}')"

    if action_name == "aggregate_sum":
        return f"aggregate_sum(column='{payload['column']}')"

    if action_name == "aggregate_mean":
        return f"aggregate_mean(column='{payload['column']}')"

    if action_name == "aggregate_count":
        return f"aggregate_count(column='{payload['column']}')"

    if action_name == "sort_by":
        return f"sort_by(column='{payload['column']}', order='{payload['order']}')"

    if action_name == "top_k":
        return f"top_k(k={payload['k']})"

    raise ValueError(f"Unsupported action: {action_name}")


def _execute_parsed_action(current_state, parsed_action, source_df):
    """Execute one parsed action using ToolExecutor's tool implementations."""
    executor = ToolExecutor(source_df)
    tool = parsed_action["tool"]
    args = parsed_action.get("args", {})

    if tool == "filter":
        return executor._filter(current_state, **args)
    if tool == "groupby":
        return executor._groupby(current_state, **args)
    if tool == "aggregate":
        return executor._aggregate(current_state, **args)
    if tool == "sort":
        return executor._sort(current_state, **args)
    if tool == "topk":
        return executor._topk(current_state, **args)
    raise ValueError(f"Unknown tool: {tool}")


def _format_observation(result_obj, max_chars=P2_OBSERVATION_CHAR_LIMIT):
    """Format an observation while handling groupby objects and long tables."""
    if result_obj is None:
        return "No data returned."

    if hasattr(result_obj, "groups") and hasattr(result_obj, "obj"):
        group_keys = getattr(result_obj, "keys", None)
        if isinstance(group_keys, (list, tuple)):
            group_label = ", ".join(str(key) for key in group_keys)
        elif group_keys is not None:
            group_label = str(group_keys)
        else:
            group_label = "the requested column(s)"
        sample_keys = list(result_obj.groups.keys())[:10]
        sample_preview = ", ".join(str(key) for key in sample_keys)
        suffix = " ..." if len(result_obj.groups) > len(sample_keys) else ""
        return (
            f"Grouped by {group_label} into {len(result_obj.groups)} groups. "
            f"Sample groups: {sample_preview}{suffix}"
        )

    if hasattr(result_obj, "empty") and result_obj.empty:
        return "Empty result."

    if getattr(result_obj, "shape", None) == (1, 1):
        return str(result_obj.iloc[0, 0])

    if isinstance(result_obj, pd.Series):
        text = result_obj.to_string()
    elif isinstance(result_obj, pd.DataFrame):
        text = result_obj.to_string(index=False)
    else:
        text = str(result_obj)

    if len(text) > max_chars:
        total_rows = getattr(result_obj, "shape", ["?"])[0]
        text = text[:max_chars] + f"\n... (truncated, {total_rows} rows total)"
    return text


def build_react_trace(entry, source_df):
    """Build a multi-step ReAct trace from a trajectory entry."""
    actions = entry["actions"]
    trace_lines = []
    current_state = source_df.copy()

    for i, action_str in enumerate(actions):
        action_name = action_str.split("(", 1)[0]
        args_text = _extract_args_text(action_str)
        action_payload = _action_payload_from_action_string(action_str)
        if action_payload is None:
            return None

        template = _THOUGHT_TEMPLATES.get(action_name, "I need to {args}.")
        thought = template.format(args=args_text)
        if i == 0:
            thought = "Let me start. " + thought

        trace_lines.append(f"Thought: {thought}")
        trace_lines.append(f"Action: {action_name}")
        trace_lines.append(
            f"Action Input: {json.dumps(action_payload, ensure_ascii=False, separators=(',', ': '))}"
        )

        parsed = parse_agent_action(action_str)
        if parsed is None:
            return None

        try:
            current_state = _execute_parsed_action(current_state, parsed, source_df)
            obs = _format_observation(current_state, max_chars=P2_OBSERVATION_CHAR_LIMIT)
        except Exception:
            return None

        trace_lines.append(f"Observation: {obs}")

    final_answer = compute_answer(entry["actions"], source_df)
    if final_answer is None:
        return None

    trace_lines.append("Thought: I now have all the information needed to answer.")
    trace_lines.append(f"Final Answer: {json.dumps(final_answer, ensure_ascii=False)}")

    return "\n".join(trace_lines)


# Build traces for all trajectories.
react_data = []
react_skipped = 0

for entry in trajectories:
    trace = build_react_trace(entry, df)
    if trace is None:
        react_skipped += 1
        continue
    react_data.append({
        "question": entry["query"],
        "trace": trace,
    })

print(f"ReAct traces built: {len(react_data)} / {len(trajectories)}")
print(f"Skipped           : {react_skipped}")
print("\n-- Sample trace --")
print(react_data[0]["trace"][:1500])


## Cell 19 — Phase 2 Dataset & Tokenizer

In [ ]:
import requests

# Load Phase 2 tokenizer
tokenizer_p2 = AutoTokenizer.from_pretrained(MODEL_NAME_P2, trust_remote_code=True)
if tokenizer_p2.pad_token is None:
    tokenizer_p2.pad_token = tokenizer_p2.eos_token
tokenizer_p2.padding_side = "right"

EOS_P2 = tokenizer_p2.eos_token
print(f"Phase 2 EOS token: {EOS_P2!r}")

TOOLALPACA_PROMPT_TEMPLATE = """### System
You are a tool-use agent. Use the provided API documentation to solve the user request.
Use exactly this format:
Thought: <reasoning>
Action: <tool_name>
Action Input: {{"arg": value}}
Observation: <tool result>
Final Answer: <answer>

### API Name
{name}

### API Category
{category}

### API Description
{description}

### API Documentation
{documentation}

### User Query
{question}

### Agent Scratchpad
"""


def format_react_example(row):
    """Format a sales-domain ReAct trace into the full training text."""
    prompt = REACT_PROMPT_TEMPLATE.format(
        system=REACT_SYSTEM_PROMPT,
        question=row["question"],
        scratchpad="",
    )
    return {"text": prompt + row["trace"] + EOS_P2}


def _toolalpaca_to_text(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    return json.dumps(value, ensure_ascii=False)


def _toolalpaca_compact_json(value):
    text = _toolalpaca_to_text(value)
    if not text:
        return "{}"
    try:
        return json.dumps(json.loads(text), ensure_ascii=False)
    except Exception:
        return text


def _toolalpaca_docs(api_entry):
    candidates = [
        api_entry.get("NLDocumentation"),
        api_entry.get("Functions"),
        api_entry.get("Introduction"),
        api_entry.get("Description"),
    ]
    for candidate in candidates:
        text = _toolalpaca_to_text(candidate)
        if text:
            return text[:P2_WARMUP_DOC_CHAR_LIMIT]
    return ""


def build_toolalpaca_example(api_entry, instance):
    steps = instance.get("intermediate_steps") or []
    if not steps:
        return None
    if len(steps) > P2_WARMUP_MAX_STEPS_PER_INSTANCE:
        return None

    question = _toolalpaca_to_text(instance.get("input", "")).split("\nHint: ", 1)[0].strip()
    tool_docs = _toolalpaca_docs(api_entry)
    if not question or not tool_docs:
        return None

    prompt = TOOLALPACA_PROMPT_TEMPLATE.format(
        name=_toolalpaca_to_text(api_entry.get("Name", "Unknown API")),
        category=_toolalpaca_to_text(api_entry.get("Category", "Unknown")),
        description=_toolalpaca_to_text(api_entry.get("Description", "")),
        documentation=tool_docs,
        question=question,
    )

    trace_lines = []
    for step in steps:
        if not step or len(step) < 2 or not step[0] or len(step[0]) < 3:
            return None

        action_name = _toolalpaca_to_text(step[0][0])
        action_input = _toolalpaca_compact_json(step[0][1])
        thought_action = _toolalpaca_to_text(step[0][2])
        thought_text = thought_action.split("\nAction:", 1)[0].strip()
        thought_text = thought_text.removeprefix("Thought:").strip()
        if not thought_text:
            thought_text = f"I should use {action_name}."

        observation = _toolalpaca_to_text(step[1])
        if len(observation) > P2_OBSERVATION_CHAR_LIMIT:
            observation = observation[:P2_OBSERVATION_CHAR_LIMIT] + "\n... (truncated)"

        trace_lines.append(f"Thought: {thought_text}")
        trace_lines.append(f"Action: {action_name}")
        trace_lines.append(f"Action Input: {action_input}")
        trace_lines.append(f"Observation: {observation}")

    final_thought = _toolalpaca_to_text(instance.get("Final Thought", "I can answer the user now."))
    final_answer = _toolalpaca_to_text(instance.get("output", ""))
    if not final_answer:
        return None

    trace_lines.append(f"Thought: {final_thought}")
    trace_lines.append(f"Final Answer: {final_answer}")

    return {"text": prompt + "\n".join(trace_lines) + EOS_P2}


# Load or cache ToolAlpaca train_data.json from the official repository.
if not os.path.isfile(P2_TOOLALPACA_CACHE_PATH):
    print(f"Downloading ToolAlpaca train data from: {P2_TOOLALPACA_URL}")
    response = requests.get(P2_TOOLALPACA_URL, timeout=120)
    response.raise_for_status()
    with open(P2_TOOLALPACA_CACHE_PATH, "wb") as f:
        f.write(response.content)
    print(f"Cached ToolAlpaca data at: {P2_TOOLALPACA_CACHE_PATH}")
else:
    print(f"Using cached ToolAlpaca data at: {P2_TOOLALPACA_CACHE_PATH}")

with open(P2_TOOLALPACA_CACHE_PATH, "r", encoding="utf-8") as f:
    toolalpaca_raw = json.load(f)

all_toolalpaca_examples = []
for api_entry in toolalpaca_raw:
    for instance in api_entry.get("Instances", []):
        example = build_toolalpaca_example(api_entry, instance)
        if example is not None:
            all_toolalpaca_examples.append(example)

random.seed(42)
random.shuffle(all_toolalpaca_examples)
p2_toolalpaca_train_examples = all_toolalpaca_examples[:P2_WARMUP_MAX_EXAMPLES]
p2_toolalpaca_train_ds = Dataset.from_list(p2_toolalpaca_train_examples)

if len(p2_toolalpaca_train_ds) == 0:
    raise RuntimeError("ToolAlpaca warm-up dataset is empty after preprocessing.")

# Shuffle and split the sales-domain ReAct data.
random.seed(42)
random.shuffle(react_data)

P2_TRAIN_SIZE = min(P2_MAX_TRAIN_EXAMPLES, int(len(react_data) * 0.9))
P2_VALID_SIZE = min(P2_MAX_VALID_EXAMPLES, len(react_data) - P2_TRAIN_SIZE)

p2_raw_train = react_data[:P2_TRAIN_SIZE]
p2_raw_valid = react_data[P2_TRAIN_SIZE : P2_TRAIN_SIZE + P2_VALID_SIZE]

p2_train_ds = Dataset.from_list(p2_raw_train).map(
    format_react_example, remove_columns=["question", "trace"]
)
p2_valid_ds = Dataset.from_list(p2_raw_valid).map(
    format_react_example, remove_columns=["question", "trace"]
)

sample_toolalpaca_len = len(tokenizer_p2(p2_toolalpaca_train_ds[0]["text"]).input_ids)
sample_sales_len = len(tokenizer_p2(p2_train_ds[0]["text"]).input_ids)

print(f"\nToolAlpaca warm-up train : {len(p2_toolalpaca_train_ds)} examples")
print(f"Warm-up sample tokens    : {sample_toolalpaca_len} (target <= {MAX_SEQ_LENGTH_P2})")
print(f"Sales ReAct train        : {len(p2_train_ds)} examples")
print(f"Sales ReAct valid        : {len(p2_valid_ds)} examples")
print(f"Sales sample tokens      : {sample_sales_len} (target <= {MAX_SEQ_LENGTH_P2})")
print("\nToolAlpaca warm-up sample (truncated):")
print(p2_toolalpaca_train_ds[0]["text"][:800])
print("\nSales ReAct sample (truncated):")
print(p2_train_ds[0]["text"][:800])


## Cell 20 — Load 7B Model + LoRA (Phase 2)

Loads Mistral-7B in 4-bit QLoRA. If a trained adapter already exists, loads it directly for inference.

In [ ]:
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Compute dtype: {compute_dtype}")

# Free Phase 1 model memory if it's still loaded
import gc
if 'model' in dir() and model is not None:
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print("Phase 1 model freed from GPU memory.")

# Phase 2 quantization config
bnb_config_p2 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

from peft import PeftModel

if P2_SKIP_TRAINING:
    print(f"Loading base model + saved Phase 2 adapter from: {P2_CKPT_PATH}")
    base_model_p2 = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME_P2,
        quantization_config=bnb_config_p2,
        device_map="auto",
        trust_remote_code=True,
    )
    model_p2 = PeftModel.from_pretrained(base_model_p2, P2_CKPT_PATH)
    model_p2.eval()
    model_p2.config.use_cache = True
    print("Phase 2 adapter loaded — model ready for inference.")
else:
    print(f"Loading {MODEL_NAME_P2} for training...")
    model_p2 = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME_P2,
        quantization_config=bnb_config_p2,
        device_map="auto",
        trust_remote_code=True,
    )
    model_p2.config.use_cache = False
    model_p2 = prepare_model_for_kbit_training(
        model_p2,
        use_gradient_checkpointing=P2_USE_GRADIENT_CHECKPOINTING,
    )

    if P2_STATE == "TIMED_CHECKPOINT" and P2_CKPT_PATH is not None:
        print(f"Loading timed Phase 2 adapter checkpoint for continued training: {P2_CKPT_PATH}")
        model_p2 = PeftModel.from_pretrained(
            model_p2,
            P2_CKPT_PATH,
            is_trainable=True,
        )
    elif P2_WARMUP_READY:
        print(f"Loading mandatory ToolAlpaca warm-up adapter from: {P2_WARMUP_ADAPTER_DIR}")
        model_p2 = PeftModel.from_pretrained(
            model_p2,
            P2_WARMUP_ADAPTER_DIR,
            is_trainable=True,
        )
    else:
        lora_config_p2 = LoraConfig(
            r=P2_LORA_R,
            lora_alpha=P2_LORA_ALPHA,
            lora_dropout=P2_LORA_DROPOUT,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=P2_LORA_TARGET_MODULES,
        )
        model_p2 = get_peft_model(model_p2, lora_config_p2)

    model_p2.print_trainable_parameters()

!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader


## Cell 21 — Phase 2 SFT Training

Fine-tunes the 7B model on ReAct-format traces. Uses the same checkpoint resume logic as Phase 1.

In [ ]:
if P2_SKIP_TRAINING:
    print(f"Phase 2 final adapter already exists: {P2_CKPT_PATH}")
    print("Skipping training because the finalized adapter is available.")
else:
    import gc
    import inspect as _inspect
    import glob as _glob
    import math as _math
    import time as _time
    from transformers import TrainerCallback

    _sft_sig_p2 = set(_inspect.signature(SFTConfig.__init__).parameters.keys())
    _trainer_sig_p2 = set(_inspect.signature(SFTTrainer.__init__).parameters.keys())

    _eval_key_p2 = "eval_strategy" if "eval_strategy" in _sft_sig_p2 else "evaluation_strategy"
    _tok_key_p2 = "processing_class" if "processing_class" in _trainer_sig_p2 else "tokenizer"

    _p2_batch = P2_BATCH_SIZE
    _p2_grad_accum = P2_GRAD_ACCUM
    _steps_per_epoch_p2 = max(1, _math.ceil(len(p2_train_ds) / (_p2_batch * _p2_grad_accum)))
    _warmup_steps_p2 = max(1, int(0.05 * _steps_per_epoch_p2 * P2_NUM_EPOCHS))
    _total_steps_p2 = max(1, int(_math.ceil(_steps_per_epoch_p2 * P2_NUM_EPOCHS)))
    _p2_save_steps = max(1, min(P2_LOGGING_STEPS, _steps_per_epoch_p2))

    def _p2_checkpoint_step(path, sep):
        try:
            return int(path.rsplit(sep, 1)[-1])
        except (TypeError, ValueError):
            return -1

    # Prefer full HF Trainer checkpoints because they include optimizer/scheduler state.
    _ckpts_p2 = [
        p for p in _glob.glob(f"{CKPT_DIR_P2}/checkpoint-*")
        if _p2_checkpoint_step(p, "-") >= 0
    ]
    _resume_p2 = None
    _latest_step_p2 = None
    _timed_step_p2 = None
    _timed_remaining_steps_p2 = None
    _continuing_from_timed_p2 = False

    if _ckpts_p2:
        _ckpts_p2.sort(key=lambda p: _p2_checkpoint_step(p, "-"))
        _resume_p2 = _ckpts_p2[-1]
        _latest_step_p2 = _p2_checkpoint_step(_resume_p2, "-")
        print(f"Found {len(_ckpts_p2)} full Trainer checkpoint(s). Highest: step {_latest_step_p2} / {_total_steps_p2}")
        print(f"Resuming Phase 2 training from checkpoint: {_resume_p2}")
    elif P2_STATE == "TIMED_CHECKPOINT" and P2_CKPT_PATH is not None:
        _continuing_from_timed_p2 = True
        _timed_step_p2 = _p2_checkpoint_step(P2_CKPT_PATH, "_")
        if _timed_step_p2 >= 0:
            _timed_remaining_steps_p2 = max(1, _total_steps_p2 - _timed_step_p2)
            print(f"No full Trainer checkpoint found. Continuing from timed adapter snapshot: {P2_CKPT_PATH}")
            print(f"Timed snapshot step: {_timed_step_p2} / {_total_steps_p2}; running {_timed_remaining_steps_p2} remaining training step(s).")
        else:
            print(f"No full Trainer checkpoint found. Continuing from timed adapter snapshot: {P2_CKPT_PATH}")
            print("Could not parse timed snapshot step; running the configured full training schedule from those adapter weights.")
        print("Timed snapshots restore adapter weights only, not optimizer/scheduler state.")
    else:
        print("No Phase 2 checkpoint found - starting fresh.")

    _cfg_extra_p2 = {}
    _trainer_extra_p2 = {}
    for _param, _val in [
        ("max_seq_length", MAX_SEQ_LENGTH_P2),
        ("dataset_text_field", "text"),
        ("packing", False),
    ]:
        if _param in _sft_sig_p2:
            _cfg_extra_p2[_param] = _val
        elif _param in _trainer_sig_p2:
            _trainer_extra_p2[_param] = _val

    def _make_sft_config(output_dir, num_train_epochs, warmup_steps, enable_eval, save_steps=None, max_steps=None):
        _common_args = dict(
            output_dir=output_dir,
            seed=42,
            num_train_epochs=num_train_epochs,
            per_device_train_batch_size=_p2_batch,
            gradient_accumulation_steps=_p2_grad_accum,
            learning_rate=P2_LEARNING_RATE,
            lr_scheduler_type="cosine",
            warmup_steps=warmup_steps,
            optim="paged_adamw_8bit",
            bf16=torch.cuda.is_bf16_supported(),
            fp16=not torch.cuda.is_bf16_supported(),
            logging_steps=P2_LOGGING_STEPS,
            save_strategy="epoch" if enable_eval else "steps",
            save_total_limit=2,
            report_to="none",
        )
        if not enable_eval:
            _common_args["save_steps"] = max(1, save_steps or P2_LOGGING_STEPS)
        if max_steps is not None:
            _common_args["max_steps"] = max(1, int(max_steps))
        if enable_eval:
            _common_args.update(
                load_best_model_at_end=True,
                metric_for_best_model="eval_loss",
            )
        return SFTConfig(
            **_common_args,
            **{_eval_key_p2: "epoch" if enable_eval else "no"},
            **_cfg_extra_p2,
        )

    if P2_WARMUP_REQUIRED and not P2_WARMUP_READY and _resume_p2 is None and not _continuing_from_timed_p2:
        _warmup_steps_per_epoch = max(1, _math.ceil(len(p2_toolalpaca_train_ds) / (_p2_batch * _p2_grad_accum)))
        _warmup_warmup_steps = max(1, int(0.05 * _warmup_steps_per_epoch * P2_WARMUP_NUM_EPOCHS))
        _warmup_save_steps = max(1, min(P2_LOGGING_STEPS, _warmup_steps_per_epoch))
        warmup_args_p2 = _make_sft_config(
            output_dir=P2_WARMUP_CKPT_DIR,
            num_train_epochs=P2_WARMUP_NUM_EPOCHS,
            warmup_steps=_warmup_warmup_steps,
            enable_eval=False,
            save_steps=_warmup_save_steps,
        )

        warmup_trainer_kwargs_p2 = dict(
            model=model_p2,
            **{_tok_key_p2: tokenizer_p2},
            args=warmup_args_p2,
            train_dataset=p2_toolalpaca_train_ds,
            **_trainer_extra_p2,
        )
        warmup_trainer_p2 = SFTTrainer(**warmup_trainer_kwargs_p2)

        print("=" * 70)
        print("PHASE 2 -- Mandatory ToolAlpaca Warm-up")
        print("=" * 70)
        print(f"Warm-up examples : {len(p2_toolalpaca_train_ds)}")
        print(f"Warm-up epochs   : {P2_WARMUP_NUM_EPOCHS}")
        print(f"Warm-up seq len  : {MAX_SEQ_LENGTH_P2}")
        print(f"Warm-up output   : {P2_WARMUP_ADAPTER_DIR}")
        warmup_trainer_p2.train()

        model_p2.save_pretrained(P2_WARMUP_ADAPTER_DIR)
        tokenizer_p2.save_pretrained(P2_WARMUP_ADAPTER_DIR)
        P2_WARMUP_READY = True
        print(f"Saved ToolAlpaca warm-up adapter to: {P2_WARMUP_ADAPTER_DIR}")

        del warmup_trainer_p2
        gc.collect()
        torch.cuda.empty_cache()
    elif P2_WARMUP_REQUIRED and P2_WARMUP_READY:
        print(f"Using existing ToolAlpaca warm-up adapter: {P2_WARMUP_ADAPTER_DIR}")
    elif P2_WARMUP_REQUIRED and (_resume_p2 is not None or _continuing_from_timed_p2):
        print("Skipping separate ToolAlpaca warm-up because a Phase 2 checkpoint already exists.")

    final_args_p2 = _make_sft_config(
        output_dir=CKPT_DIR_P2,
        num_train_epochs=P2_NUM_EPOCHS,
        warmup_steps=_warmup_steps_p2,
        enable_eval=P2_ENABLE_TRAIN_EVAL,
        save_steps=_p2_save_steps,
        max_steps=_timed_remaining_steps_p2 if _resume_p2 is None else None,
    )

    class TimedCheckpointCallbackP2(TrainerCallback):
        """Saves adapter weights to Drive every `interval_min` minutes."""

        def __init__(self, adapter_dir, interval_min=5):
            self.adapter_dir = adapter_dir
            self.interval_sec = interval_min * 60
            self.last_save = _time.time()

        def on_step_end(self, args, state, control, model=None, **kwargs):
            elapsed = _time.time() - self.last_save
            if elapsed >= self.interval_sec:
                save_path = f"{self.adapter_dir}/timed_ckpt_step_{state.global_step}"
                os.makedirs(save_path, exist_ok=True)
                model.save_pretrained(save_path)
                tokenizer_p2.save_pretrained(save_path)
                self.last_save = _time.time()
                print()
                print(f"[P2 TimedCheckpoint] Saved at step {state.global_step} "
                      f"to {save_path} ({elapsed/60:.1f} min)")

    trainer_kwargs_p2 = dict(
        model=model_p2,
        **{_tok_key_p2: tokenizer_p2},
        args=final_args_p2,
        train_dataset=p2_train_ds,
        callbacks=[TimedCheckpointCallbackP2(ADAPTER_DIR_P2, interval_min=5)],
        **_trainer_extra_p2,
    )
    if P2_ENABLE_TRAIN_EVAL:
        trainer_kwargs_p2["eval_dataset"] = p2_valid_ds

    trainer_p2 = SFTTrainer(**trainer_kwargs_p2)

    print("=" * 70)
    print("PHASE 2 -- Sales Tool Fine-Tuning")
    print("=" * 70)
    print(f"Sales examples        : {len(p2_train_ds)}")
    print(f"Validation examples   : {len(p2_valid_ds)}")
    print(f"Epochs                : {P2_NUM_EPOCHS}")
    print(f"Batch / grad accum    : {_p2_batch} / {_p2_grad_accum}")
    print(f"Max sequence length   : {MAX_SEQ_LENGTH_P2}")
    print(f"Train-time eval       : {P2_ENABLE_TRAIN_EVAL}")
    print(f"Trainer save steps    : {_p2_save_steps}")
    print(f"LoRA targets          : {P2_LORA_TARGET_MODULES} | r={P2_LORA_R} | alpha={P2_LORA_ALPHA}")

    trainer_p2.train(resume_from_checkpoint=_resume_p2)


## Cell 22 — Save Phase 2 Adapter

In [ ]:
if not P2_SKIP_TRAINING:
    model_p2.save_pretrained(ADAPTER_DIR_P2)
    tokenizer_p2.save_pretrained(ADAPTER_DIR_P2)
    print(f"Phase 2 adapter saved to: {ADAPTER_DIR_P2}")
else:
    print(f"Phase 2 adapter was already saved at: {P2_CKPT_PATH}")

print("\nPhase 2 adapter directory contents:")
for fname in sorted(os.listdir(ADAPTER_DIR_P2)):
    fpath = os.path.join(ADAPTER_DIR_P2, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  {fname:<40s}  {size_mb:.2f} MB")
    else:
        print(f"  {fname:<40s}  [directory]")

## Cell 23 — ReAct Execution Engine (Part B)

Pure Python implementation of the ReAct agent loop. No LangChain/LlamaIndex.

Key features:
1. **Custom StoppingCriteria** — halts generation at "Observation:" to prevent hallucinated tool outputs
2. **Robust JSON parsing** — sanitizes malformed JSON (trailing commas, markdown blocks)
3. **State management** — truncates large observations to prevent OOM
4. **Bounded error recovery** — max 2 consecutive errors before graceful termination
5. **Final Answer detection** — breaks loop when model outputs "Final Answer:"

In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList


class ObservationStopCriteria(StoppingCriteria):
    """Halt generation as soon as the model starts emitting an Observation."""

    def __init__(self, tokenizer, triggers=("Observation:",)):
        self.tokenizer = tokenizer
        self.triggers = triggers

    def __call__(self, input_ids, scores, **kwargs):
        tail = self.tokenizer.decode(input_ids[0, -40:], skip_special_tokens=True)
        return any(trigger in tail for trigger in self.triggers)


def _sanitize_json(text):
    """Best-effort cleanup for malformed JSON from smaller models."""
    text = text.strip()
    text = re.sub(r"```(?:json)?\s*", "", text)
    text = re.sub(r"```", "", text)
    text = text.replace("\u201c", '"').replace("\u201d", '"').replace("\u2019", "'")
    text = re.sub(r",\s*([}\]])", r"\1", text)
    return text.strip()


def _extract_json_object(text):
    """Extract the first balanced JSON object from text."""
    text = _sanitize_json(text)
    start = text.find("{")
    if start == -1:
        return text

    depth = 0
    in_string = False
    escape = False

    for idx in range(start, len(text)):
        ch = text[idx]
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return text[start : idx + 1]

    return text[start:]


class ReActAgent:
    """Pure-Python ReAct execution engine for the fine-tuned 7B model."""

    def __init__(self, model, tokenizer, source_df, max_steps=5, max_obs_chars=P2_OBSERVATION_CHAR_LIMIT,
                 max_consecutive_errors=2, max_new_tokens=256):
        self.model = model
        self.tokenizer = tokenizer
        self.source_df = source_df
        self.max_steps = max_steps
        self.max_obs_chars = max_obs_chars
        self.max_consecutive_errors = max_consecutive_errors
        self.max_new_tokens = max_new_tokens
        self.stop_criteria = StoppingCriteriaList([ObservationStopCriteria(tokenizer)])

    def run(self, question):
        """Execute the ReAct loop for a given question."""
        self.model.eval()
        self.model.config.use_cache = True

        scratchpad_parts = []
        consecutive_errors = 0
        all_actions = []
        current_state = self.source_df.copy()

        for step in range(self.max_steps):
            prompt = REACT_PROMPT_TEMPLATE.format(
                system=REACT_SYSTEM_PROMPT,
                question=question,
                scratchpad="".join(scratchpad_parts),
            )

            output_text = self._generate(prompt).strip()
            if output_text:
                scratchpad_parts.append(output_text)
                if not output_text.endswith("\n"):
                    scratchpad_parts.append("\n")

            if "Final Answer:" in output_text:
                answer_text = output_text.split("Final Answer:", 1)[-1].strip()
                eos = self.tokenizer.eos_token or ""
                answer_text = answer_text.replace(eos, "").strip()
                return {
                    "actions": all_actions,
                    "answer": self._parse_final_answer(answer_text),
                    "steps": step + 1,
                    "scratchpad": "".join(scratchpad_parts),
                }

            action_name, action_input = self._parse_action(output_text)
            if action_name is None:
                consecutive_errors += 1
                obs = "ERROR - Could not parse action. Use Action plus JSON Action Input."
                scratchpad_parts.append(f"Observation: {obs}\n")
                if consecutive_errors >= self.max_consecutive_errors:
                    return {
                        "actions": all_actions,
                        "answer": None,
                        "error": f"Max consecutive errors ({self.max_consecutive_errors}) reached.",
                        "steps": step + 1,
                        "scratchpad": "".join(scratchpad_parts),
                    }
                continue

            try:
                action_payload = self._parse_action_input_payload(action_name, action_input)
                action_str = _action_string_from_payload(action_name, action_payload)
                parsed = parse_agent_action(action_str)
                if parsed is None:
                    raise ValueError(f"Unknown action: {action_str}")

                all_actions.append(action_str)
                current_state = _execute_parsed_action(current_state, parsed, self.source_df)
                obs = _format_observation(current_state, max_chars=self.max_obs_chars)
                consecutive_errors = 0
            except Exception as exc:
                consecutive_errors += 1
                obs = f"ERROR - {type(exc).__name__}: {exc}"

            scratchpad_parts.append(f"Observation: {obs}\n")

            if consecutive_errors >= self.max_consecutive_errors:
                return {
                    "actions": all_actions,
                    "answer": None,
                    "error": f"Max consecutive errors ({self.max_consecutive_errors}) reached.",
                    "steps": step + 1,
                    "scratchpad": "".join(scratchpad_parts),
                }

        return {
            "actions": all_actions,
            "answer": None,
            "error": f"Max steps ({self.max_steps}) reached without Final Answer.",
            "steps": self.max_steps,
            "scratchpad": "".join(scratchpad_parts),
        }

    def _generate(self, prompt):
        """Generate text from the model, stopping before Observation text."""
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SEQ_LENGTH_P2,
        ).to(self.model.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                stopping_criteria=self.stop_criteria,
            )

        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

    def _parse_action(self, text):
        """Extract Action and Action Input from generated text."""
        action_match = re.search(r"Action:\s*([A-Za-z_][\w]*)", text)
        input_match = re.search(r"Action Input:\s*(.*)", text, re.DOTALL)

        if not action_match or not input_match:
            return None, None

        action_input = input_match.group(1).strip()
        if "Observation:" in action_input:
            action_input = action_input.split("Observation:", 1)[0].rstrip()
        return action_match.group(1).strip(), action_input

    def _parse_action_input_payload(self, action_name, action_input_text):
        """Parse JSON Action Input, with a fallback for legacy syntax."""
        cleaned = action_input_text.strip()
        json_candidate = _extract_json_object(cleaned)

        if json_candidate.startswith("{"):
            payload = json.loads(_sanitize_json(json_candidate))
            return _normalize_action_payload(action_name, payload)

        legacy_action = f"{action_name}({cleaned})"
        payload = _action_payload_from_action_string(legacy_action)
        if payload is None:
            raise ValueError(f"Could not parse Action Input for {action_name}: {action_input_text}")
        return _normalize_action_payload(action_name, payload)

    def _parse_final_answer(self, text):
        """Robustly parse the Final Answer text."""
        text = _sanitize_json(text)

        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass

        obj_candidate = _extract_json_object(text)
        if obj_candidate.startswith("{"):
            try:
                return json.loads(_sanitize_json(obj_candidate))
            except json.JSONDecodeError:
                pass

        list_match = re.search(r"\[.*\]", text, re.DOTALL)
        if list_match:
            try:
                return json.loads(_sanitize_json(list_match.group()))
            except json.JSONDecodeError:
                pass

        num_match = re.search(r"-?[\d]+(?:\.[\d]+)?", text)
        if num_match:
            val = num_match.group()
            return float(val) if "." in val else int(val)

        return text


print("ReActAgent class defined.")
print("ObservationStopCriteria: halts at 'Observation:'")
print("Ready for inference.")


## Cell 24 — ReAct Agent Demo & Evaluation

Runs the ReAct agent on test queries, showing the full multi-turn trace and comparing against ground truth.

In [ ]:
agent = ReActAgent(model_p2, tokenizer_p2, df, max_steps=5)

react_test_queries = [
    {
        "question": "What is the total revenue for 2022?",
        "expected_actions": ["filter_data(column='year', value=2022)", "aggregate_sum(column='revenue')"],
    },
    {
        "question": "Which city had the highest profit in 2021? Top 1",
        "expected_actions": ["filter_data(column='year', value=2021)", "group_by(column='city')", "aggregate_sum(column='profit')", "sort_by(column='profit', order='desc')", "top_k(k=1)"],
    },
    {
        "question": "What is the average revenue by city?",
        "expected_actions": ["group_by(column='city')", "aggregate_mean(column='revenue')"],
    },
    {
        "question": "List top 3 cities by revenue in 2022.",
        "expected_actions": ["filter_data(column='year', value=2022)", "group_by(column='city')", "aggregate_sum(column='revenue')", "sort_by(column='revenue', order='desc')", "top_k(k=3)"],
    },
    {
        "question": "What is the total profit for 2023?",
        "expected_actions": ["filter_data(column='year', value=2023)", "aggregate_sum(column='profit')"],
    },
]

print("=" * 70)
print("PHASE 2 -- ReAct Agent Evaluation")
print("=" * 70)

for entry in react_test_queries:
    q = entry["question"]
    expected_actions = entry["expected_actions"]
    expected_answer = compute_answer(expected_actions, df)

    result = agent.run(q)
    model_actions = result.get("actions", [])
    model_answer = result.get("answer")
    executor_answer = compute_answer(model_actions, df) if model_actions else None
    steps = result.get("steps", "?")
    error = result.get("error")

    print(f"\nQ: {q}")
    print(f"  Steps               : {steps}")
    print(f"  Expected actions    : {expected_actions}")
    print(f"  Model actions       : {model_actions}")
    print(f"  Expected answer     : {expected_answer}")
    print(f"  Model final answer  : {model_answer}")
    print(f"  Executor answer     : {executor_answer}")
    if error:
        print(f"  Error               : {error}")
    print(f"  Action match        : {normalize_answer(executor_answer) == normalize_answer(expected_answer)}")
    print(f"  Final answer match  : {normalize_answer(model_answer) == normalize_answer(expected_answer)}")
    print("-" * 70)

    if q == react_test_queries[0]["question"]:
        print("\n  -- Full Scratchpad (first query) --")
        print(result.get("scratchpad", "")[:2000])
        print("  -- End Scratchpad --")
        print("-" * 70)


## Cell 25 — (Bonus) Direct Preference Optimization (DPO)

DPO training on top of the SFT model. Creates preference pairs:
- **Chosen**: Correct ReAct traces (from training data)
- **Rejected**: Perturbed traces (wrong column names, wrong action order, hallucinated outputs)

Uses `trl.DPOTrainer` with the SFT model as the reference model.

In [ ]:
# -- Bonus: DPO Training --
# This cell creates preference pairs and runs DPO on the Phase 2 SFT model.

DPO_ADAPTER_DIR = f"{DRIVE_DIR}/mistral-react-dpo-adapter-fast-nogc"
DPO_CKPT_DIR = f"{DRIVE_DIR}/mistral-react-dpo-ckpt-fast-nogc"
os.makedirs(DPO_ADAPTER_DIR, exist_ok=True)
os.makedirs(DPO_CKPT_DIR, exist_ok=True)

# -- Step 1: Generate rejected traces by perturbing correct ones --

WRONG_COLUMNS = ["invalid_col", "nonexistent", "revenue2", "total", "amount"]
VALID_COLUMNS = ["date", "year", "month", "city", "region", "product", "category",
                 "revenue", "units_sold", "cost", "profit"]


def perturb_trace(trace_text):
    """Create a rejected version of a ReAct trace by introducing errors."""
    lines = trace_text.split("\n")
    perturbed = []
    changed = False

    for line in lines:
        if line.startswith("Action Input:") and random.random() < 0.5:
            # Replace a valid column with a wrong one.
            for col in VALID_COLUMNS:
                if col in line:
                    wrong = random.choice(WRONG_COLUMNS)
                    line = line.replace(col, wrong, 1)
                    changed = True
                    break
        elif line.startswith("Thought:") and random.random() < 0.3:
            # Add conversational filler; the trained model should avoid this.
            line = "Thought: Sure! I'd be happy to help with that! " + line[9:]
            changed = True
        perturbed.append(line)

    # Ensure the rejected trace is different from the chosen trace.
    if not changed:
        for i, line in enumerate(perturbed):
            if line.startswith("Action Input:"):
                for col in VALID_COLUMNS:
                    if col in line:
                        perturbed[i] = line.replace(col, WRONG_COLUMNS[0], 1)
                        changed = True
                        break
                if changed:
                    break

    if not changed:
        for i, line in enumerate(perturbed):
            if line.startswith("Thought:"):
                perturbed[i] = "Thought: Sure! I'd be happy to help with that! " + line[9:]
                break

    return "\n".join(perturbed)


# Build DPO dataset
dpo_data = []
for item in react_data[:500]:  # Use a subset for DPO
    prompt = REACT_PROMPT_TEMPLATE.format(
        system=REACT_SYSTEM_PROMPT,
        question=item["question"],
        scratchpad="",
    )
    chosen = item["trace"]
    rejected = perturb_trace(item["trace"])

    dpo_data.append({
        "prompt": prompt,
        "chosen": chosen + EOS_P2,
        "rejected": rejected + EOS_P2,
    })

dpo_ds = Dataset.from_list(dpo_data)
print(f"DPO preference pairs: {len(dpo_ds)}")
print("\n-- Sample chosen (first 300 chars) --")
print(dpo_ds[0]["chosen"][:300])
print("\n-- Sample rejected (first 300 chars) --")
print(dpo_ds[0]["rejected"][:300])


# -- Step 2: DPO Training --
import gc
import inspect as _inspect

from peft import PeftModel
from trl import DPOConfig, DPOTrainer

_dpo_cfg_sig = set(_inspect.signature(DPOConfig.__init__).parameters.keys())
_dpo_trainer_sig = set(_inspect.signature(DPOTrainer.__init__).parameters.keys())
_dpo_tok_key = "processing_class" if "processing_class" in _dpo_trainer_sig else "tokenizer"

_dpo_cfg_extra = {}
_dpo_trainer_extra = {}
for _param, _value in [
    ("max_length", MAX_SEQ_LENGTH_P2),
    ("max_prompt_length", MAX_SEQ_LENGTH_P2 // 2),
    ("max_completion_length", MAX_SEQ_LENGTH_P2 // 2),
]:
    if _param in _dpo_cfg_sig:
        _dpo_cfg_extra[_param] = _value
    elif _param in _dpo_trainer_sig:
        _dpo_trainer_extra[_param] = _value

# If Phase 2 was loaded from an existing adapter, reload it in trainable mode for DPO.
if P2_SKIP_TRAINING:
    print(f"Reloading Phase 2 adapter for DPO from: {P2_CKPT_PATH}")
    del model_p2
    gc.collect()
    torch.cuda.empty_cache()

    _base_model_dpo = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME_P2,
        quantization_config=bnb_config_p2,
        device_map="auto",
        trust_remote_code=True,
    )
    _base_model_dpo.config.use_cache = False
    _base_model_dpo = prepare_model_for_kbit_training(
        _base_model_dpo, use_gradient_checkpointing=True
    )
    model_p2 = PeftModel.from_pretrained(
        _base_model_dpo,
        P2_CKPT_PATH,
        is_trainable=True,
    )
else:
    model_p2.config.use_cache = False

model_p2.train()
if hasattr(model_p2, "print_trainable_parameters"):
    model_p2.print_trainable_parameters()

dpo_config = DPOConfig(
    output_dir=DPO_CKPT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    optim="paged_adamw_8bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    beta=0.1,
    **_dpo_cfg_extra,
)

dpo_trainer = DPOTrainer(
    model=model_p2,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_ds,
    **{_dpo_tok_key: tokenizer_p2},
    **_dpo_trainer_extra,
)

print(f"DPO tokenizer key: {_dpo_tok_key!r}")
print(f"DPO params -> DPOConfig: {_dpo_cfg_extra} | DPOTrainer: {_dpo_trainer_extra}")
print("Starting DPO training...")
dpo_trainer.train()

# Save DPO adapter
model_p2.save_pretrained(DPO_ADAPTER_DIR)
tokenizer_p2.save_pretrained(DPO_ADAPTER_DIR)
print(f"\nDPO adapter saved to: {DPO_ADAPTER_DIR}")